# Planetary Analogs — בדיקת דאטה לפני מידול
הריצו את כל התאים לפי הסדר. ברירת המחדל מורידה את המאגר המורחב מ־GitHub. אפשר להעלות ל־Colab את העותק המקומי דרך חלונית Files ולהגדיר את הנתיב שלו ב־`LOCAL_DB`. [מקורות ומילון שדות](https://github.com/romimana/Planetary-analogs/blob/main/ML.md).

In [ ]:
import hashlib
import json
import sqlite3
from contextlib import closing
from pathlib import Path
from urllib.request import urlopen

import pandas as pd
from IPython.display import display

DATA_REF = "26111d1ebffb6573c43e5fb7559edf59394af3d9"  # Set to "main" for the latest published data.
LOCAL_DB = None  # Example: "/content/ml-full-local.sqlite"

if LOCAL_DB:
    db_path = Path(LOCAL_DB).resolve()
else:
    base = f"https://raw.githubusercontent.com/romimana/Planetary-analogs/{DATA_REF}/data"
    with urlopen(f"{base}/ml-full-coverage.json", timeout=60) as response:
        expected_hash = json.load(response)["github"]["sha256"]
    with urlopen(f"{base}/ml-full.sqlite", timeout=120) as response:
        content = response.read()
    if hashlib.sha256(content).hexdigest() != expected_hash:
        raise ValueError("Checksum mismatch. Re-run this cell or use a pinned DATA_REF.")
    db_path = Path("ml-full.sqlite").resolve()
    db_path.write_bytes(content)

with closing(sqlite3.connect(db_path.as_uri() + "?mode=ro", uri=True)) as db:
    objects = db.execute(
        "SELECT name, type FROM sqlite_schema WHERE type IN ('table', 'view') "
        "AND name NOT LIKE 'sqlite_%' ORDER BY type, name"
    ).fetchall()
    tables = {
        name: pd.read_sql_query('SELECT * FROM "' + name.replace('"', '""') + '"', db)
        for name, _ in objects
    }

inventory = pd.DataFrame([
    {"name": name, "type": kind, "rows": len(tables[name]), "columns": tables[name].shape[1]}
    for name, kind in objects
])
display(inventory)

## תצוגה וחוסרים
שנו את `TABLE` ורוצו שוב. שמות הטבלאות מופיעים בתא הקודם. אפשר לגשת לכל טבלה גם דרך `tables["name"]`.

In [ ]:
TABLE = "earth_profiles"
df = tables[TABLE]
display(df.head(8))
missing = df.isna().sum()
display(pd.DataFrame({
    "dtype": df.dtypes.astype(str),
    "missing": missing,
    "missing_pct": (missing / max(len(df), 1) * 100).round(1),
}).sort_values("missing", ascending=False))

## בחירת שדות משותפים
ברירת המחדל מדגימה בחירה של חמישה רכיבי כימיה, כך שגם יפן יכולה להיכנס. הפעילו סיליקה אם דרושה השוואה של רכיב מרכזי בחומרים סיליקטיים. סינון החוסרים חל רק על השדות שנבחרו. אפשר לבחור גם מדדי שטח מאותו חלון 5×5 ק״מ.

In [ ]:
BODY = "mars"  # "mars" or "moon"
INCLUDE_SILICA = False
INCLUDE_TERRAIN = False

CHEMISTRY = {
    "sio2_wt_pct": "Silica — a major component of silicate-rich material",
    "tio2_wt_pct": "Titanium dioxide — helps distinguish titanium-rich material",
    "mgo_wt_pct": "Magnesium oxide — helps characterize magnesium-rich minerals",
    "cao_wt_pct": "Calcium oxide — helps characterize calcium-rich material",
    "k2o_wt_pct": "Potassium oxide — helps distinguish potassium-rich material",
    "mno_wt_pct": "Manganese oxide — captures differences in manganese content",
}
TERRAIN = [
    "slope_median_deg", "slope_p90_deg", "slope_over_10_deg_fraction",
    "relief_p90_p10_m", "local_roughness_rms_m",
]
features = [name for name in CHEMISTRY if INCLUDE_SILICA or name != "sio2_wt_pct"]
if INCLUDE_TERRAIN:
    features += TERRAIN

if BODY not in {"mars", "moon"}:
    raise ValueError("BODY must be 'mars' or 'moon'")
earth = tables["earth_profiles"]
planet = tables["planet_samples"]
targets = planet.loc[(planet["body"] == BODY) & (planet["comparison_status"] != "chemistry_hold")]

earth_model = earth.dropna(subset=features)[["id", "site_id", "source_id", "material", *features]].copy()
target_model = targets.dropna(subset=features)[["id", "area", "location_role", "material", *features]].copy()
coverage = pd.concat([
    earth.groupby("source_id").size().rename("all_samples"),
    earth_model.groupby("source_id")["site_id"].agg(samples="size", sites="nunique"),
], axis=1).fillna(0).astype(int)
display(coverage)
display(earth_model.head())
display(target_model.head())
X_earth = earth_model[features].copy()
X_targets = target_model[features].copy()

## תרחישים סביבתיים
טבלאות `complete_earth`, `complete_mars`, `complete_moon` הן נקודת הפתיחה כשצריך תרחיש מלא. הטמפרטורות של MODIS, REMS ו־Diviner תלויות בשיטת המדידה. עיינו בשם השדה ובתיעוד לפני השוואה. טבלאות המקור משמרות מדידות אוויר ופני שטח עם חותמות זמן נפרדות. שמרו תרחישים מאותו אתר יחד בזמן הערכה.

In [ ]:
earth_scenarios = tables["complete_earth"]
target_scenarios = tables[f"complete_{BODY}"]
display(earth_scenarios[["site_id", "date", "pass", "air_temperature_k",
                         "temperature_k", "pressure_pa", "weather_distance_m"]].head())
temperature = "ground_brightness_k" if BODY == "mars" else "lunar_surface_bolometric_k"
display(target_scenarios[["id", "window_label", "local_solar_hour", "distance_m", temperature]].head())